In [ ]:
# Classifying potential ocean current regions

# Gulf Stream
gulf_stream_mask = (
    (mean_speed > 0.5) &
    (mean_uo > 0) &
    (mean_vo > 0)
)

# Canary Current
canary_mask = (
    (mean_speed > 0.1) &
    (mean_vo < 0) &
    (mean_speed.latitude >= 15) &
    (mean_speed.latitude <= 35) &
    (mean_speed.longitude >= -20) &
    (mean_speed.longitude <= -5)
)

# North Equatorial Current
north_equatorial_mask = (
    (mean_speed > 0.2) &
    (mean_uo < 0) &
    (mean_speed.latitude >= 10) &
    (mean_speed.latitude <= 25)
)

# Remove overlap with the Canary Current
north_equatorial_mask = north_equatorial_mask & ~canary_mask

# Azores Current
azores_mask = (
    (mean_speed > 0.1) &
    (mean_uo > 0) &
    (mean_speed.latitude >= 30) &
    (mean_speed.latitude <= 40) &
    (mean_speed.longitude >= -40) &
    (mean_speed.longitude <= -20)
)

# Create a classification array
classification = xr.full_like(mean_speed, 0, dtype=int)

classification = xr.where(gulf_stream_mask, 1, classification)
classification = xr.where(north_equatorial_mask, 2, classification)
classification = xr.where(canary_mask, 3, classification)
classification = xr.where(azores_mask, 4, classification)

# Count grid points in each region
masks = {
    "Gulf Stream": gulf_stream_mask,
    "North Equatorial": north_equatorial_mask,
    "Canary": canary_mask,
    "Azores": azores_mask
}

for name, mask in masks.items():
    grid_points = int(mask.sum().values)
    average_speed = float(mean_speed.where(mask).mean().values)

    print(name)
    print("Grid points:", grid_points)
    print("Average speed:", round(average_speed, 3), "m/s")
    print()